# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tanzimul3islam/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

**Lane 2: Refresh / Content Opportunity Scoring.** This continues the provisional lane,
500-impression eligibility floor, and 20-page review budget in
[Week 1](w01_research_question.ipynb). It frames a task and checks a baseline; it does not train a model.

**Run:** Python 3 with pandas and a Jupyter kernel (Colab includes both). Run all cells in order.
The loader finds the bundled CSV from the repo or notebook directory, or fetches the same
public file at a pinned revision in Colab. Its checksum prevents silently changing the sample.

Guidance: [framing skill](../../skills/framing-ml-problems/SKILL.md),
[FlyRank data skill](../../skills/flyrank/flyrank-data/SKILL.md),
[data dictionary](../../docs/data-dictionary.md), and
[lane guide](../../docs/ml-intern-dataset-and-lane-guide.md).


## 1. My lane as an ML task (type)

**Task type: ranking / scoring.** For a content editor or SEO lead deciding which visible pages
to review first, I will build a ranked queue using exposure, freshness, and content context.
The intended score ranks the likelihood that an independent review will find a justified
content-refresh action. The practical outcome is better use of a limited review budget,
measured by precision@20. A binary classifier could supply the score later, but the user-facing
task is selecting the first 20 pages, rather than assigning every page an automatic action.

**Action:** An editor inspects the selected pages for outdated facts, missing coverage, or an
intent mismatch; they also check seasonality, technical problems, and traffic shifting to related
pages. They refresh only when the evidence supports a content change, otherwise monitor or
route the issue for investigation. The eventual queue should include a score, reason codes,
and an evidence-coverage flag. That flag is not a promise that editing will recover traffic.

**Wrong-call costs:** A false positive wastes a review slot and can prompt an unnecessary edit.
A false negative delays attention to a page that needs work. Neither editor hours nor revenue
effects are measured in this CSV, so I do not assign a monetary benefit.

**ML loop:** decision and review capacity → load and verify page-level data → collect independent
review outcomes → compare a simple baseline with a learned ranking → evaluate on held-out
clients → give the editor a reasoned queue → record decisions and later outcomes → monitor
precision and coverage before revising the system. This notebook completes framing and a
same-snapshot metric check; label collection, training, and validation remain future work.


In [1]:
from pathlib import Path
from io import BytesIO
from urllib.request import urlopen
import hashlib
import platform
import pandas as pd

DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")
DATA_COMMIT = "86863f1578555c069a9f4afea00fbdce6037bf75"
EXPECTED_SHA256 = "c43bdac4eccfa17fcd8a33974fe36f2c998c03a3ae3af8d80cf712abab5d6396"
local_file = next((root / DATA_PATH for root in [Path.cwd(), *Path.cwd().parents]
                   if (root / DATA_PATH).is_file()), None)
if local_file is not None:
    raw = local_file.read_bytes()
    source = "Bundled starter CSV (local checkout)"
else:
    # Colab often opens a notebook without cloning the repository.
    url = ("https://raw.githubusercontent.com/tanzimul3islam/"
           f"flyrank-ml-internship-starter/{DATA_COMMIT}/{DATA_PATH.as_posix()}")
    with urlopen(url, timeout=60) as response:
        raw = response.read()
    source = "Public starter CSV (pinned GitHub revision)"

assert hashlib.sha256(raw).hexdigest() == EXPECTED_SHA256, "Dataset changed; review the framing numbers."
df = pd.read_csv(BytesIO(raw))
assert df.shape == (30000, 44)
assert df["content_id"].notna().all() and df["content_id"].is_unique
assert df["client_id"].notna().all() and df["client_id"].nunique() == 32
print(source)
print(f"Grain verified: {len(df):,} unique content items, {df.client_id.nunique()} pseudonymized clients.")
print(f"Python {platform.python_version()}; pandas {pd.__version__}")
print(f"CSV SHA-256: {EXPECTED_SHA256}")
from IPython.display import display

# Provisional policies inherited from Week 1, not optimized on these results.
MIN_IMPRESSIONS = 500
K = 20
GOAL_EDITORIAL_PRECISION = 0.70
MIN_ABSOLUTE_LIFT = 0.10
required = ["impressions_90d", "days_since_last_update", "trend_direction"]
assert df[required].notna().all().all()
assert df["impressions_90d"].ge(0).all()
assert df["days_since_last_update"].ge(0).all()
lane = df.loc[df["impressions_90d"].ge(MIN_IMPRESSIONS)].copy()
assert lane["content_id"].is_unique
assert not lane.duplicated(["client_id", "content_id"]).any()
assert len(lane) >= K
print(f"Lane slice: {len(lane):,} / {len(df):,} pages; {lane.client_id.nunique()} clients.")
print("One row = one pseudonymized content item at the export snapshot; no report date is supplied.")


Bundled starter CSV (local checkout)
Grain verified: 30,000 unique content items, 32 pseudonymized clients.
Python 3.14.6; pandas 3.0.6
CSV SHA-256: c43bdac4eccfa17fcd8a33974fe36f2c998c03a3ae3af8d80cf712abab5d6396
Lane slice: 16,726 / 30,000 pages; 28 clients.
One row = one pseudonymized content item at the export snapshot; no report date is supplied.


## 2. Target or proxy

**Desired target: `editor_confirmed_refresh`**, a nullable binary column. An independent editor
records **1** if inspection confirms a specific, justified content-refresh action, **0** if the
review finds no refresh action, and **missing** until review is complete or if evidence is
insufficient. Review notes should document the issue and proposed action. Reviewers should not
see model scores or the proxy label, and disagreements should be adjudicated. This is an
observed human judgment, not an if-statement applied to traffic. It still does not measure
the causal benefit of editing. These judgments are absent from the starter CSV; every value
below is genuinely missing rather than fabricated.

**Available descriptive proxy: `observed_decline_proxy`**, 1 for the supplied
`trend_direction == "down"`, otherwise 0. The dictionary defines down as a greater-than-20%
impression drop from the previous 30 days to the last 30 days. This is a threshold-derived
summary of observed traffic, **not** an observed editorial decision, future decline, or
refresh response. A 0 can include a page with no previous-window exposure; it does not mean
the page is healthy. The proxy is useful only for sketching a target and checking metric code.

I will not train a model to reproduce this definition and call that editorial intelligence.
`trend_direction`, `trend_pct`, the constructed proxy, and either target column must not be
predictors. IDs are for grouping/joining only. Raw 30-day comparison inputs and overlapping
90-day totals also need a window audit: withholding the named label alone would not make
a future-prediction experiment leakage-free.

For a later forecasting extension, build features strictly before a decision date and observe
impression change in a non-overlapping later window, using daily history and adequate client
coverage. The starter snapshot cannot establish that temporal test. For the primary editorial
task, collect real review outcomes before attempting supervised learning.


In [2]:
assert lane["trend_direction"].isin(["new", "flat", "up", "down", "stable"]).all()
lane["observed_decline_proxy"] = lane["trend_direction"].eq("down").astype("Int64")
lane["editor_confirmed_refresh"] = pd.Series(pd.NA, index=lane.index, dtype="Int64")
assert lane["observed_decline_proxy"].isin([0, 1]).all()
assert lane["editor_confirmed_refresh"].isna().all()

target_summary = pd.DataFrame({
    "target_column": ["observed_decline_proxy", "editor_confirmed_refresh"],
    "meaning": ["Recorded down flag: descriptive only", "Independent editorial judgment: not collected"],
    "known_labels": [lane.observed_decline_proxy.notna().sum(), lane.editor_confirmed_refresh.notna().sum()],
    "positive_labels": [lane.observed_decline_proxy.sum(), pd.NA],
    "missing_labels": [lane.observed_decline_proxy.isna().sum(), lane.editor_confirmed_refresh.isna().sum()],
})
display(target_summary)
print(f"No prior-window impressions: {lane.impressions_prev_30d.eq(0).sum():,} eligible pages.")


,target_column,meaning,known_labels,positive_labels,missing_labels
0,observed_decline_proxy,Recorded down flag: descriptive only,16726,9961,0
1,editor_confirmed_refresh,Independent editorial judgment: not collected,0,<NA>,16726


No prior-window impressions: 109 eligible pages.


## 3. Success metric

**Primary metric: editorial precision@20 = confirmed refresh actions in the first 20 / 20.**
Twenty is the provisional capacity from Week 1, not a measured stakeholder constraint.
All 20 must have completed reviews; missing judgments are not zeros and must not be silently
dropped. Precision matches the cost of wasting scarce review slots, though it does not measure
how many actionable pages the queue misses.

**Proposed definition of good:** at least **0.70 (14 of 20)** independently confirmed pages,
and at least **0.10 absolute improvement (two extra useful pages per 20)** over the freshness
baseline on the same evaluation pool. These are planning targets for stakeholder confirmation,
not measured achievements. A single 20-page queue is too small to establish reliable lift;
report results and uncertainty across multiple review cycles and clients.

**Comparator:** Among pages with at least 500 impressions/90d, rank longest-unupdated first,
breaking ties by impressions, then a stable pseudonym for reproducibility. The pseudonym
has no predictive meaning. A random queue's expected proxy precision is the eligible pool's
proxy prevalence. Neither comparator filters on the proxy outcome.

The code below measures **proxy precision@20 only** on this already-inspected snapshot.
It is a computation check, not a held-out result or evidence of ML lift. Before training,
reserve clients for evaluation, tune only within development clients, and freeze the method
before scoring the held-out pool. Review the union of both methods' top-20 selections under
the same rubric and without showing scores. Report client representation and per-client
results because a pooled queue can hide poor performance on smaller clients. If repeated
snapshots are introduced, also separate feature and outcome windows in time.


In [3]:
def precision_at_k(ranked, target, k):
    assert len(ranked) >= k, "Need a full review queue."
    labels = ranked.head(k)[target]
    assert labels.notna().all(), "Precision is unavailable until all top-K labels are observed."
    assert labels.isin([0, 1]).all()
    return float(labels.sum() / k)

baseline = lane.sort_values(
    ["days_since_last_update", "impressions_90d", "content_id"],
    ascending=[False, False, True], kind="stable",
)
proxy_p20 = precision_at_k(baseline, "observed_decline_proxy", K)
proxy_base_rate = float(lane["observed_decline_proxy"].mean())
display(pd.DataFrame([
    {"check": "Freshness baseline: descriptive proxy precision@20", "value": proxy_p20},
    {"check": "Random queue: expected proxy precision@20", "value": proxy_base_rate},
    {"check": "Editorial precision@20: labels not collected", "value": float("nan")},
]))
print(f"Top {K}: {int(baseline.head(K).observed_decline_proxy.sum())} recorded-down pages.")
print(f"Baseline minus random expectation: {proxy_p20 - proxy_base_rate:+.2%} (proxy only).")
print("Editorial success cannot be assessed yet; no model was trained or evaluated.")


,check,value
0,Freshness baseline: descriptive proxy precisio...,0.80000
1,Random queue: expected proxy precision@20,0.59554
2,Editorial precision@20: labels not collected,NaN


Top 20: 16 recorded-down pages.
Baseline minus random expectation: +20.45% (proxy only).
Editorial success cannot be assessed yet; no model was trained or evaluated.


## 4. The unit of analysis, as a real dataframe

**One row = one content item (page) at the starter export snapshot**, not a query, visit,
client, or daily observation. `content_id` is unique in the full 30,000-row CSV; the
client–content pair is also unique in the lane slice. The slice includes every page with
at least 500 trailing-90-day impressions, with no outcome-based filtering. This intentionally
excludes low-exposure pages; conclusions apply only to the eligible pool.

Below are five actual rows, in original CSV order, with a compact set of measurements and both
target columns. `source_row` is only the original dataframe row index for this display; it is
not a feature or durable page identifier. Pseudonymous IDs remain in memory for grouping and
later client-holdout validation. No private names, page URLs, titles, or queries are shown.

The candidate input dataframe illustrates exposure, freshness, content type, and content length.
It retains missing word counts and adds an availability flag instead of replacing unknowns
with zero. `avg_position == 0` is converted to missing with a coverage flag; rate columns such
as CTR are percentages, not fractions. These are candidate inputs for a later editorial task,
not a certified feature set for future forecasting. No imputation, fitting, or random split
is needed at this framing stage.


In [4]:
candidate_features = lane[[
    "content_type", "impressions_90d", "days_since_last_update", "word_count", "avg_position"
]].copy()
candidate_features["has_word_count"] = candidate_features["word_count"].notna()
candidate_features["avg_position"] = candidate_features["avg_position"].mask(
    candidate_features["avg_position"].eq(0)
)
candidate_features["has_position"] = candidate_features["avg_position"].notna()

forbidden = {"content_id", "client_id", "trend_direction", "trend_pct",
             "observed_decline_proxy", "editor_confirmed_refresh"}
assert forbidden.isdisjoint(candidate_features.columns)
assert len(candidate_features) == len(lane)
unit_preview = candidate_features.join(
    lane[["observed_decline_proxy", "editor_confirmed_refresh"]]
).head(5).rename_axis("source_row")
display(unit_preview)
print(f"Lane dataframe: {len(lane):,} rows, each a distinct page.")
print(f"Candidate inputs: {candidate_features.shape}; both targets are stored separately.")
print("<NA> in editor_confirmed_refresh means not reviewed, not a negative result.")


,content_type,impressions_90d,days_since_last_update,word_count,avg_position,has_word_count,has_position,observed_decline_proxy,editor_confirmed_refresh
source_row,,,,,,,,,
0,keyword article,3803,20,3221.0,10.6,True,True,1,<NA>
1,keyword article,15320,25,2481.0,20.3,True,True,1,<NA>
2,keyword article,12581,20,3515.0,36.5,True,True,1,<NA>
3,keyword article,11751,22,NaN,6.2,False,True,0,<NA>
4,keyword article,19140,14,2803.0,44.0,True,True,1,<NA>


Lane dataframe: 16,726 rows, each a distinct page.
Candidate inputs: (16726, 7); both targets are stored separately.
<NA> in editor_confirmed_refresh means not reviewed, not a negative result.


## 5. Why ML beats a fixed rule here

**ML has not beaten a rule here yet.** If the only task were to identify the supplied down
flag, its fixed definition would already solve it. My real task is choosing pages for useful
editorial review, for which that flag is only incomplete evidence.

A fixed freshness rule ignores interactions: an old, stable reference page may need no edit,
while a recently updated page may still miss the reader's intent. Exposure, content type,
keyword context, and missing measurements can change what a freshness signal means. Learning
from independent editorial outcomes could improve the ranking if those patterns recur across
clients, but this is a hypothesis. None of these correlations would prove a refresh causes
traffic recovery. Useful content context is also absent from this CSV and must come from the
review workflow.

The executable baseline below checks how many clients it represents and the proxy mix within
freshness groups. These are descriptive diagnostics, not editorial labels or causal effects.
I will keep the simple baseline if a more complex method does not meet the predeclared
editorial precision and lift goals under held-out evaluation. Until review labels exist,
this is an analysis-backed prioritization problem, not a ready supervised-ML application.


In [5]:
freshness_check = lane.assign(
    freshness_group=lane["days_since_last_update"].ge(90).map(
        {True: "Unupdated >= 90 days", False: "Updated < 90 days"}
    )
).groupby("freshness_group").agg(
    pages=("content_id", "size"),
    recorded_down_pages=("observed_decline_proxy", "sum"),
    recorded_down_share=("observed_decline_proxy", "mean"),
)
display(freshness_check)
print(f"Baseline top-{K} represents {baseline.head(K).client_id.nunique()} of {lane.client_id.nunique()} eligible clients.")
print("Groups use the provisional Week 1 90-day threshold; shares describe the proxy, not refresh value.")


,pages,recorded_down_pages,recorded_down_share
freshness_group,,,
Unupdated >= 90 days,6575,4053,0.616426
Updated < 90 days,10151,5908,0.582012


Baseline top-20 represents 5 of 28 eligible clients.
Groups use the provisional Week 1 90-day threshold; shares describe the proxy, not refresh value.


## 6. Self-check

- [x] Named Lane 2 and ranking/scoring as the task; linked it to the ML loop.
- [x] Named the editor, real content action, review capacity, and costs of wrong calls.
- [x] Distinguished the missing editorial target from the threshold-derived decline proxy.
- [x] Defined precision@20 and a prospective success threshold before evaluating any model.
- [x] Loaded the actual starter slice, checked uniqueness, and displayed a five-row dataframe with target columns.
- [x] Computed a transparent baseline's proxy metric without claiming future predictive performance.
- [x] Explained where ML might help and why a rule remains the comparator.
- [x] Ran every code cell top to bottom and saved executed outputs with no errors.
- [x] Kept private client names, page URLs, and queries out of the notebook; displayed only a small approved starter-data preview and summaries.
- [x] Used observed, directional, and decision-support claims; made no causal refresh claim.

**Deliverable:** Commit this executed notebook at `work/notebooks/w02_ml_task_framing.ipynb`,
push the commit, and submit the repository URL on the assignment card. Portal submission
is a separate step; this notebook does not claim it has happened.

**AI assistance:** An AI assistant drafted and executed this notebook using the repo's framing
and data skills. Lane and capacity continue Week 1; success thresholds are provisional proposals
for the intern to review. No stakeholder interview, editorial labeling, model validation, or
refresh experiment is claimed.
